In [ ]:
# Outpatient Trend Analysis

## Phase 5

Feature Engineering

Purpose

Create analytical variables that improve reporting,
SQL analysis and Power BI dashboards.

In [13]:
import pandas as pd 
import numpy as np
import os

print("Libraries loaded successfully")

Libraries loaded successfully


In [14]:
CLEAN_PATH = "../Data/cleaned"

PROCESSED_PATH = "../Data/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Directories verified.")

Directories verified.


In [15]:
df_departments = pd.read_csv(
    f"{CLEAN_PATH}/departments_clean.csv"
)

df_diagnoses = pd.read_csv(
    f"{CLEAN_PATH}/diagnoses_clean.csv"
)

df_providers = pd.read_csv(
    f"{CLEAN_PATH}/providers_clean.csv"
)

df_patients = pd.read_csv(
    f"{CLEAN_PATH}/patients_clean.csv"
)

df_visits = pd.read_csv(
    f"{CLEAN_PATH}/visits_clean.csv"
)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [16]:
df_patients["date_of_birth"] = pd.to_datetime(
    df_patients["date_of_birth"]
)

df_patients["registration_date"] = pd.to_datetime(
    df_patients["registration_date"]
)

df_visits["visit_date"] = pd.to_datetime(
    df_visits["visit_date"]
)

print("Date columns converted.")

Date columns converted.


In [19]:
today = pd.Timestamp.today()
df_patients["age"] = (
    (today - df_patients["date_of_birth"]).dt.days
    //365
)
print(df_patients[["age"]].head())


   age
0   73
1    3
2   17
3   35
4   22


In [20]:
# Age Categories

def age_category(age):
    if age <=12:
        return "Child"
    elif age <=17:
        return "Adolescent"
    elif age <=59:
        return "Adult"
    else:
        return "Elderly"

df_patients["age_category"] = (df_patients["age"].apply(age_category)
                              )
df_patients["age_category"].value_counts()

age_category
Adult         2427
Child         1307
Elderly        975
Adolescent     291
Name: count, dtype: int64

In [21]:
# Visit Month

df_visits["visit_month"] = (df_visits["visit_date"].dt.month_name())
df_visits["visit_year"] = (df_visits["visit_date"].dt.year)
df_visits["visit_quarter"]= (df_visits["visit_date"].dt.quarter)

print("Calender features created")

Calender features created


In [22]:
# Weekday
df_visits["weekday"] = (df_visits["visit_date"].dt.day_name())

In [26]:
# Visit Hour
df_visits["arrival_time"] = (
    pd.to_datetime(df_visits["arrival_time"])
      .dt.strftime("%H:%M:%S")
)

df_visits["arrival_hour"] = pd.to_datetime(
    df_visits["arrival_time"],
    format="%H:%M:%S"
).dt.hour

print("Arrival hour extracted.")

C:\Users\USER\AppData\Local\Temp\ipykernel_13000\3957218601.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(df_visits["arrival_time"])


Arrival hour extracted.


In [27]:
# Peak hour indicator
df_visits["peak_period"] = np.where(

    df_visits["arrival_hour"].between(7,10),

    "Peak Hours",

    "Off Peak"

)

df_visits["peak_period"].value_counts()

peak_period
Peak Hours    14350
Off Peak       5650
Name: count, dtype: int64

In [28]:
# Waiting Time Category

def waiting_category(minutes):

    if minutes <= 15:
        return "Short"

    elif minutes <= 30:
        return "Moderate"

    elif minutes <= 60:
        return "Long"

    return "Critical"

df_visits["waiting_category"] = (

    df_visits["waiting_time_minutes"]

    .apply(waiting_category)

)

In [29]:
# Revenue Category

def revenue_band(amount):
    if amount <100:
        return "Low"
    elif amount <2500:
        return "Medium"
    elif amount<4000:
        return "High"
    else :
        return "Very High"
df_visits["revenue_band"] = (df_visits["bill_amount"]
                             .apply(revenue_band)
                            )

In [30]:
# Consultation Category
def consultation_level(minutes):
    if minutes<=10:
        return "Short"
    elif minutes<=20:
        return "Standard"
    else:
        return "Extended"
df_visits["consultation_level"] = (df_visits["consultation_minutes"]
                                   .apply(consultation_level)
                                  )

In [21]:
print("="*70)

print("PHASE 5 COMPLETE")

print("="*70)

print("✓ Patient Age")

print("✓ Age Category")

print("✓ Visit Month")

print("✓ Visit Year")

print("✓ Visit Quarter")

print("✓ Weekday")

print("✓ Arrival Hour")

print("✓ Peak Hour")

print("✓ Waiting Category")

print("✓ Revenue Band")

print("✓ Consultation Level")

print("✓ Processed CSV Exported")

print("\nReady for Phase 6 : Validation")

PHASE 5 COMPLETE
✓ Patient Age
✓ Age Category
✓ Visit Month
✓ Visit Year
✓ Visit Quarter
✓ Weekday
✓ Arrival Hour
✓ Peak Hour
✓ Waiting Category
✓ Revenue Band
✓ Consultation Level
✓ Processed CSV Exported

Ready for Phase 6 : Validation


In [31]:
# Insurance Classification
# INSURANCE GROUP

# Standardize values
df_patients["insurance_type"] = (
    df_patients["insurance_type"]
        .astype(str)
        .str.strip()
)

# Mapping based on ACTUAL dataset values
insurance_map = {

    "SHA": "Government",

    "Private Insurance": "Private",

    "Cash": "Self Pay",

    "Corporate Insurance": "Corporate"

}

df_patients["insurance_group"] = (
    df_patients["insurance_type"]
        .map(insurance_map)
)

print("\nInsurance Type Distribution")
print(df_patients["insurance_type"].value_counts())

print("\nInsurance Group Distribution")
print(df_patients["insurance_group"].value_counts())

print("\nMissing Values")
print(df_patients["insurance_group"].isnull().sum())


Insurance Type Distribution
insurance_type
SHA                    2823
Private Insurance       948
Cash                    735
Corporate Insurance     494
Name: count, dtype: int64

Insurance Group Distribution
insurance_group
Government    2823
Private        948
Self Pay       735
Corporate      494
Name: count, dtype: int64

Missing Values
0


In [32]:
print(df_patients["insurance_type"].value_counts(dropna=False))

insurance_type
SHA                    2823
Private Insurance       948
Cash                    735
Corporate Insurance     494
Name: count, dtype: int64


In [33]:
# Registration Year
df_patients["registration_year"] = (df_patients["registration_date"].dt.year
                                   )
df_patients["registration_month"] = (df_patients["registration_date"].dt.month_name()
                                    )
print("Registration timeline created")

Registration timeline created


In [34]:
# Weekend Indicator
# WEEKEND FLAG
df_visits["weekend"] = np.where(
    df_visits["weekday"].isin([
        "Saturday",
        "Sunday"
    ]),
    "Weekend",
    "Weekday"
)
df_visits["weekend"].value_counts()

weekend
Weekday    14188
Weekend     5812
Name: count, dtype: int64

In [35]:
# Financial Year
def financial_year(date):

    if date.month >=7:

        return f"{date.year}/{date.year+1}"

    return f"{date.year-1}/{date.year}"

df_visits["financial_year"]=(

df_visits["visit_date"]

.apply(financial_year)

)

print(df_visits["financial_year"].value_counts())

financial_year
2024/2025    9605
2025/2026    7122
2023/2024    3273
Name: count, dtype: int64


In [36]:
# Season
def season(month):
    if month in[3,4,5]:
        return"Long Rains"
    elif month in [10,11,12]:
        return "Short Rains"
    return "Dry Season"
df_visits["season"] = (df_visits["visit_date"].dt.month
                       .apply(season)
                      )

In [37]:
# Clinic Workload
def clinic_load(wait):
    if wait<=20:
        return "Low"
    elif wait<=40:
        return "Moderate"
    elif wait<=60:
        return "High"
    return "Critical"
df_visits["clinic_load"]=(df_visits["waiting_time_minutes"].apply(clinic_load)
                         )


In [38]:
# High Cost visits (Flag)
threshold = df_visits["bill_amount"].quantile(.90)
df_visits["high_cost_visit"]=np.where(df_visits["bill_amount"]>=threshold,
                                      "Yes",
                                      "No"
                                     )
print(df_visits["high_cost_visit"].value_counts())

high_cost_visit
No     18000
Yes     2000
Name: count, dtype: int64


In [39]:
# Repeat Patient Flag
visit_counts= (
    df_visits.groupby("patient_id").size()
)
repeat_ids = visit_counts[visit_counts>1].index
df_visits["repeat_patient"]=np.where(df_visits["patient_id"].isin(repeat_ids),
                                     "Yes",
                                     "No"
                                    )

In [40]:
# Chronic Disease Flag
chronic_ids=[
    3,
    4,
    17
]
df_visits["chronic_case"]=np.where(df_visits["diagnosis_id"].isin(chronic_ids),
                                   "Yes",
                                   "No"
                                  )

In [41]:
print(df_visits.head())

print(df_visits.columns)

   visit_id  patient_id  provider_id  department_id  diagnosis_id visit_date  \
0         1         522            1              1            18 2025-09-25   
1         2         755           22              5            11 2024-03-22   
2         3        3847            1              1            23 2025-02-15   
3         4        2911           22              5            11 2024-07-07   
4         5        3596           37              8             2 2025-08-15   

  arrival_time  waiting_time_minutes  consultation_minutes  bill_amount  ...  \
0     11:32:38                    61                    13      3994.97  ...   
1     07:58:12                    33                    10      1110.35  ...   
2     11:26:02                    79                     5      1385.54  ...   
3     09:40:23                    32                    15       992.76  ...   
4     12:51:39                    21                    15      3238.90  ...   

  waiting_category revenue_band  consu

In [42]:
df_visits["arrival_time"] = pd.to_datetime(
    df_visits["arrival_time"],
    format="%H:%M:%S"
).dt.strftime("%H:%M:%S")

In [43]:
df_departments.to_csv(
f"{PROCESSED_PATH}/departments_processed.csv",
index=False
)

df_diagnoses.to_csv(
f"{PROCESSED_PATH}/diagnoses_processed.csv",
index=False
)

df_providers.to_csv(
f"{PROCESSED_PATH}/providers_processed.csv",
index=False
)

df_patients.to_csv(
f"{PROCESSED_PATH}/patients_processed.csv",
index=False
)

df_visits.to_csv(
f"{PROCESSED_PATH}/visits_processed.csv",
index=False
)

print("Processed datasets exported.")

Processed datasets exported.
